# Component transport on a DFN with OpenGeoSys and PorePy

This is work in progress. We thank the PorePy developers. 

In [16]:
import numpy as np
import porepy as pp
import ogstools as ot
import ogs as ogs
import os as os

## Exemplary results

<img src="img/Materials.png" style="display:block; margin-bottom:10px;">
<img src="img/pressure.png" style="display:block; margin-bottom:10px;">
<img src="img/concentration.png" style="display:block;">

# Setting up the domain and generating a random set of circular fractures

In [2]:
mins = np.array([0.,0.,0.])
maxs = np.array([100,100.,100.])

In [3]:
bounding_box = {'xmin': mins[0], 'xmax': maxs[0], 'ymin': mins[1], 'ymax': maxs[1], 'zmin': mins[2], 'zmax': maxs[2]}
domain = pp.Domain(bounding_box=bounding_box)
domain

pp.Domain(bounding_box={'xmin': np.float64(0.0), 'xmax': np.float64(100.0), 'ymin': np.float64(0.0), 'ymax': np.float64(100.0), 'zmin': np.float64(0.0), 'zmax': np.float64(100.0)})

In [14]:
nfracs = 40
r_range = np.array([30,50])
f_i = np.array([])
for i in range(nfracs):
    center = np.random.rand(3) * (maxs - mins) + mins
    major_axis = np.random.rand() * (r_range[1] - r_range[0]) + r_range[0]
    minor_axis = major_axis.copy() #circular
    major_axis_angle = 0. #for circular
    strike_angle = np.random.rand() * np.pi - np.pi/2
    dip_angle = np.random.rand() * np.pi - np.pi/2
    f_i = np.append(f_i,pp.create_elliptic_fracture(center, major_axis, minor_axis, major_axis_angle, strike_angle, dip_angle))

In [15]:
network = pp.create_fracture_network(fractures=f_i,domain=domain)
network

Three-dimensional fracture network with 40 plane fractures.
The domain is a cuboid with bounding box: {'xmin': np.float64(0.0), 'xmax': np.float64(100.0), 'ymin': np.float64(0.0), 'ymax': np.float64(100.0), 'zmin': np.float64(0.0), 'zmax': np.float64(100.0)}.

## Meshing ... 

In [16]:
mesh_args = {'cell_size_boundary': 10.0, 'cell_size_fracture': 5.0, 'cell_size_min': 1.0}
mdg = pp.create_mdg("simplex", mesh_args, network)

In [17]:
#Removal of 3D not needed really
mdg2d = mdg.copy()
for sd in mdg2d.subdomains():
    if sd.dim == 3:
        mdg2d.remove_subdomain(sd)
mdg2d

Mixed-dimensional grid containing 578 grids and 1444 interfaces.
Maximum dimension present: 2 
Minimum dimension present: 0 
40 grids of dimension 2 with in total 60259 cells
446 grids of dimension 1 with in total 2449 cells
92 grids of dimension 0 with in total 92 cells
892 interfaces between grids of dimension 2 and 1 with in total 9796 mortar cells.
552 interfaces between grids of dimension 1 and 0 with in total 552 mortar cells.

In [18]:
#pp.plot_grid(mdg2d, figsize=(12,12), plot_2d=False)

## Export to VTU and import in OGS. Setting up Material IDs

In [19]:
pp.Exporter(mdg2d, 'mixed_dimensional_grid').write_vtu()

In [2]:
DFN_2D = ot.Mesh('mixed_dimensional_grid_constant_2.vtu')
DFN_2D

Mesh (0x75f0bc61e3e0)
  N Cells:    60259
  N Points:   29372
  X Bounds:   -5.551e-17, 1.000e+02
  Y Bounds:   -8.882e-16, 1.000e+02
  Z Bounds:   0.000e+00, 1.000e+02
  N Arrays:   1

In [3]:
if 'MaterialIDs' not in DFN_2D.cell_data.keys():
    DFN_2D['MaterialIDs'] = (DFN_2D['subdomain_id'] - DFN_2D['subdomain_id'].min()).astype(np.int32)
for i in DFN_2D.cell_data.keys():
    if i != 'MaterialIDs':
        DFN_2D.cell_data.remove(i)
for i in DFN_2D.point_data.keys():
        DFN_2D.point_data.remove(i)
DFN_2D

Mesh (0x75f0bc61e3e0)
  N Cells:    60259
  N Points:   29372
  X Bounds:   -5.551e-17, 1.000e+02
  Y Bounds:   -8.882e-16, 1.000e+02
  Z Bounds:   0.000e+00, 1.000e+02
  N Arrays:   1

In [4]:
DFN_2D.save('mixed_dimensional_grid_constant_2.vtu')

In [9]:
fig = DFN_2D.plot(scalars='MaterialIDs',
    show_edges=True,
    #clim=[0, nfracs],
    show_axes=True,
    scalar_bar_args={'title': 'MaterialIDs', 'vertical': True});

Widget(value='<iframe src="http://localhost:43555/index.html?ui=P_0x75f09c58aae0_3&reconnect=auto" class="pyvi…

## Generating boundaries for OGS

In [24]:
ogs.cli.reviseMesh(i='mixed_dimensional_grid_constant_2.vtu',o='temp.vtu')
ogs.cli.NodeReordering(i='temp.vtu',o='mixed_dimensional_grid_constant_2.vtu')
ogs.cli.checkMesh('test2.vtu',**{'v': True})

[2025-03-30 11:44:27.620] [ogs] [info] Mesh read: 37126 nodes, 60259 elements.
[2025-03-30 11:44:27.620] [ogs] [info] Simplifying the mesh...
[2025-03-30 11:44:27.698] [ogs] [warning] Property MaterialIDs exists but does not have the requested mesh item type node.
[2025-03-30 11:44:27.698] [ogs] [warning] Property MaterialIDs exists but does not have the requested type f.
[2025-03-30 11:44:27.698] [ogs] [warning] Property MaterialIDs exists but does not have the requested type d.
[2025-03-30 11:44:27.775] [ogs] [info] Revised mesh: 29372 nodes, 60259 elements.
[2025-03-30 11:44:27.869] [ogs] [info] Reordering nodes... 
[2025-03-30 11:44:27.873] [ogs] [info] Corrected 30059 elements.
[2025-03-30 11:44:27.912] [ogs] [info] VTU file written.
[2025-03-30 11:44:27.924] [ogs] [error] File 'test2.vtu' does not exist.


1

In [25]:
ogs.cli.ExtractBoundary(i = 'mixed_dimensional_grid_constant_2.vtu', o='boundaries.vtu')

[2025-03-30 11:44:28.774] [ogs] [info] Mesh read: 29372 nodes, 60259 elements.
[2025-03-30 11:44:28.777] [ogs] [info] 1 property vectors copied, 0 vectors skipped.
[2025-03-30 11:44:28.777] [ogs] [info] Created surface mesh: 3406 nodes, 3459 elements.


0

In [26]:
tol = 1e-1
ogs.cli.removeMeshElements(i='boundaries.vtu',o='xmax.vtu',**{"x-max": maxs[0]-tol})
ogs.cli.removeMeshElements(i='boundaries.vtu',o='xmin.vtu',**{"x-min": mins[0]+tol})
#
#ogs.cli.removeMeshElements(i='boundaries.vtu',o='ymax.vtu',**{"y-max": maxs[1]-tol})
#ogs.cli.removeMeshElements(i='boundaries.vtu',o='ymin.vtu',**{"y-min": mins[1]+tol})
#
#ogs.cli.removeMeshElements(i='boundaries.vtu',o='zmax.vtu',**{"z-max": maxs[2]-tol})
#ogs.cli.removeMeshElements(i='boundaries.vtu',o='zmin.vtu',**{"z-min": mins[2]+tol})

[2025-03-30 11:44:29.576] [ogs] [info] Mesh read: 3406 nodes, 3459 elements.
[2025-03-30 11:44:29.576] [ogs] [info] Bounding box of "boundaries" is
x = [-0.000000,100.000000]
y = [-0.000000,100.000000]
z = [0.000000,100.000000]
[2025-03-30 11:44:29.576] [ogs] [info] 3220 elements found.
[2025-03-30 11:44:29.576] [ogs] [info] Removing total 3220 elements...
[2025-03-30 11:44:29.576] [ogs] [info] 239 elements remain in mesh.
[2025-03-30 11:44:29.576] [ogs] [info] Removing total 3166 nodes...
[2025-03-30 11:44:29.599] [ogs] [info] Mesh read: 3406 nodes, 3459 elements.
[2025-03-30 11:44:29.599] [ogs] [info] Bounding box of "boundaries" is
x = [-0.000000,100.000000]
y = [-0.000000,100.000000]
z = [0.000000,100.000000]
[2025-03-30 11:44:29.599] [ogs] [info] 3227 elements found.
[2025-03-30 11:44:29.599] [ogs] [info] Removing total 3227 elements...
[2025-03-30 11:44:29.599] [ogs] [info] 232 elements remain in mesh.
[2025-03-30 11:44:29.600] [ogs] [info] Removing total 3171 nodes...


0

In [27]:
#ogs.cli.identifySubdomains(s=tol,m='mixed_dimensional_grid_constant_2.vtu', **{'': 'xmax.vtu', '': 'xmin.vtu'})

## Input file manipulation and simulations

In [28]:
model=ot.Project(input_file='DFN_HC.prj',output_file='DFN_HC_concrete.prj')
model.write_input()
cmd = "sed 's/all_fracs/"+ ",".join(str(i) for i in range(nfracs)) + "/g' DFN_HC_concrete.prj -i"
os.system(cmd)

0

In [29]:
model.run_model()

OGS finished with project file DFN_HC_concrete.prj.
Execution took 179.5396749973297 s
Project file written to output.


## Results

In [6]:
ms = ot.MeshSeries('DFN_HC.pvd')
ms[-1]

Mesh (0x75f0b738ad40)
  N Cells:    60259
  N Points:   29372
  X Bounds:   -5.551e-17, 1.000e+02
  Y Bounds:   -8.882e-16, 1.000e+02
  Z Bounds:   0.000e+00, 1.000e+02
  N Arrays:   9

In [12]:
fig = ms[-1].plot(scalars='pressure',
    show_edges=True,
    #clim=[0, 7],
    show_axes=True,
    scalar_bar_args={'title': 'Pressure', 'vertical': True});

Widget(value='<iframe src="http://localhost:43555/index.html?ui=P_0x75f0b740d790_6&reconnect=auto" class="pyvi…

In [13]:
fig = ms[-1].plot(scalars='Si',
    show_edges=True,
    #clim=[0, 7],
    show_axes=True,
    scalar_bar_args={'title': 'Concentration', 'vertical': True})

Widget(value='<iframe src="http://localhost:43555/index.html?ui=P_0x75f09c589940_7&reconnect=auto" class="pyvi…